# Create Uehara Memorial Foundation Awards (上原記念生命科学財団)

Creates Uehara Memorial Foundation grants from the foundation's research report
search system (研究報告書検索システム, https://www.ueharazaidan.or.jp/report_search/),
which holds the final report of every research grant (研究助成金), research
encouragement grant (研究奨励金), special research-promotion grant (研究推進特別奨励金)
and overseas fellowship (海外留学助成金) funded since FY1985. An empty search returns all
of them (25/page); each hit gives theme, report year, researcher + affiliation, kind,
field, and the report PDF. **6,220 grants, grant years 1985-2023, 100% title/PI/affiliation,
98.4% grant year** (local run 2026-09-30). Japanese as published.

Not covered: grants whose reports are not yet in the system (FY2024-2025 grantees, which
exist only as per-year PDF lists at /grant/grantor.html), and the non-report kinds
(来日研究生助成金 visiting-researcher and 国際シンポジウム開催助成金 symposium grants).

**Prerequisites:**
- Run `scripts/local/uehara_foundation_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/uehara_foundation/uehara_foundation_projects.parquet`
  (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** for reports up to report year 2017 (4,527 grants) the report
PDF id IS the foundation's grant number, `{grant year}{kind digit}{serial}` (e.g.
`201320273`; 1 研究奨励, 2 研究助成, 3/4 海外留学, 9 研究推進特別奨励) - the form citing works
use (113 distinct 9-digit Uehara ids among the ~1,040 acknowledgement shells; 9 collapse
exactly onto these rows, the rest are FY2018+ or non-report grants). From report year 2018
the PDF id is a report serial (`{report year}0{serial}`) and the grant number is not
published, so those 1,693 ship as synthetic `UEHARA-RPT-{pdf id}` (prefixed so they can
never equal a real grant number); 8 hits without a PDF ship as `UEHARA-{year}-{sha1}`.

**Dates:** grant year only. Pre-2018 reports: the grant number's first 4 digits. 2018+
reports: looked up by researcher name in the foundation's own per-year grantee-list PDFs
(FY2015+, same kind, 1-4 years before the report; typically 2); no unambiguous match ->
NULL (100 rows). `start_date` = Jan 1 of the grant year (runbook §2.3 year-only pattern).
**Amounts:** not published per grant (fixed per kind and year), so NULL - §6.7 waived.

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320310760`,
metadata read from `openalex.funders.funders` (ror 00gc20a07, doi 10.13039/100008732).

**Priority:** `507` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.uehara_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/uehara_foundation/uehara_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants FROM openalex.awards.uehara_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.uehara_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.uehara_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320310760 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320310760;

## Step 2: Create Uehara Memorial Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.uehara_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320310760  -- Uehara Memorial Foundation
),

g AS (
    SELECT
        r.*,
        TRY_CAST(r.grant_year AS INT) AS yr,
        NULLIF(TRIM(r.lead_given_name), '') AS given,
        NULLIF(TRIM(r.lead_family_name), '') AS family
    FROM openalex.awards.uehara_foundation_raw r
    WHERE r.funder_award_id IS NOT NULL AND TRIM(r.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    NULLIF(TRIM(g.funding_type), '') as funding_type,
    NULLIF(TRIM(g.kind), '') as funder_scheme,
    'uehara_report_search' as provenance,
    CASE WHEN g.yr IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.yr, '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.yr as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    CASE WHEN g.family IS NOT NULL THEN named_struct(
        'given_name', g.given,
        'family_name', g.family,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.affiliation), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (§2.1.1): real grant numbers (report year <= 2017), UEHARA-RPT-{report id}
-- (2018+), UEHARA-{year}-{hash} (no PDF). Zero rows in any other shape.
SELECT
    CASE WHEN funder_award_id RLIKE '^(19|20)[0-9]{2}[1-49][0-9]{4}$' AND LEFT(funder_award_id, 4) = CAST(start_year AS STRING) THEN 'grant_number'
         WHEN funder_award_id RLIKE '^UEHARA-RPT-20[0-9]{2}0[0-9]{4}$' THEN 'report_id'
         WHEN funder_award_id RLIKE '^UEHARA-([0-9]{4}|NA)-[0-9a-f]{10}$' THEN 'synthetic'
         ELSE 'BAD' END AS form,
    COUNT(*) AS n, COUNT(DISTINCT id) AS distinct_ids, MIN(start_year) AS min_sy, MAX(start_year) AS max_sy
FROM openalex.awards.uehara_foundation_awards
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'uehara_report_search' AND priority = 507;

-- Insert into openalex_awards_raw with priority.
-- Priority 507: direct-from-funder ingest of the Uehara Memorial Foundation report search.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    507 as priority
FROM openalex.awards.uehara_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.uehara_foundation_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt
FROM openalex.awards.uehara_foundation_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.uehara_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.family_name AS family, lead_investigator.given_name AS given, COUNT(*) AS n
FROM openalex.awards.uehara_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (expect 100% title/PI, ~98% year; amount waived).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    MIN(amount) as min_amount, MAX(amount) as max_amount, AVG(amount) as avg_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.uehara_foundation_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'uehara_report_search'
GROUP BY provenance, priority;